<a href="https://colab.research.google.com/github/timraiswell/ai-engineer/blob/main/03-rag/00-what-is-rag.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# What Is RAG?

**Goal:** Understand what Retrieval-Augmented Generation is, the problems it solves, and the retrieve → augment → generate loop. Then run the whole thing in ~15 lines before the rest of the section builds each piece properly.

Part of [ai-engineer-notebooks](https://github.com/calmrocks/ai-engineer-notebooks), the hands-on companion to the [FDE / AI Engineer transition plan](https://www.calm.rocks/resources/career-development/transition-fde-ai-engineer/).

## Setup

Each notebook is self-contained, so the next two cells stand it up from scratch:

1. **Install dependencies.** The `aien` package (this repo) carries the shared setup helper and pulls in the `groq` client, the only dependency this notebook needs.
2. **Load your API key.** Get a free key at [console.groq.com](https://console.groq.com/) (no credit card). In Colab, add it via the **key icon** in the left sidebar → **Add new secret**, name it exactly `GROQ_API_KEY`, paste the value, and toggle **Notebook access** on. Running locally instead? Set `GROQ_API_KEY` as an environment variable.

(Full walkthrough and model-picking guidance live in [00-setup/00-environment.ipynb](https://colab.research.google.com/github/calmrocks/ai-engineer-notebooks/blob/main/00-setup/00-environment.ipynb).)

In [1]:
%pip install -q "git+https://github.com/calmrocks/ai-engineer-notebooks.git"

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.8/143.8 kB 1.2 MB/s eta 0:00:00


In [2]:
from aien import setup

# Loads GROQ_API_KEY (Colab Secrets or local env var) and returns a ready
# Groq client. Pass model=... to override the default; if a call later 404s,
# list available models — see 00-setup/00-environment.ipynb.
client, MODEL = setup()

Groq client ready. MODEL = openai/gpt-oss-120b


## What RAG is

A language model only knows what was in its training data. Ask it about your company's internal runbook, last week's incident, or any private document, and it will either refuse or, worse, invent a confident, wrong answer.

**Retrieval-Augmented Generation (RAG)** fixes this with one move: *before* asking the model to answer, you **retrieve** relevant text from an external source and **paste it into the prompt** as context. The model then answers from what you handed it, not from memory alone. Retrieve → augment the prompt → generate.

The original 2020 paper (Lewis et al.) framed it as combining two kinds of memory:

- **Parametric memory —** what's baked into the model's weights during training (fixed, general, no citations).
- **Non-parametric memory —** an external store you can update any time (your docs, a database, an API) that the model reads at answer time.

RAG is just the plumbing that connects the two. That's the whole idea; everything else in this section is *how to do each step well*.

## Why RAG: the four problems it solves

A plain LLM call has four failure modes that RAG directly addresses:

| Problem with a plain LLM | How RAG fixes it |
|---|---|
| **Knowledge cutoff:** training data is frozen; it doesn't know recent events | Retrieve from a source you keep current, no retraining |
| **Hallucination:** it answers confidently even when it doesn't know | Ground the answer in retrieved evidence it can quote |
| **No private/domain data:** it never saw your internal docs | Supply those docs as context at query time |
| **No citations:** you can't tell where an answer came from | Answers point back to the retrieved passages |

And a fifth, practical one: **cost**. Injecting knowledge by *fine-tuning* or retraining is expensive and slow; RAG adds knowledge by just... putting it in the prompt. That's why RAG is usually the first thing you reach for when a model needs to know something it wasn't trained on. (The full fine-tune-vs-RAG-vs-prompt decision, and why fine-tuning is the *wrong* tool for knowledge, is section 06, [Adapting the model](https://colab.research.google.com/github/calmrocks/ai-engineer-notebooks/blob/main/06-adaptation/01-fine-tune-vs-rag-vs-prompt.ipynb).)

**When *not* to use RAG:** if the task needs no external facts (summarize this email, rewrite this paragraph), retrieval adds latency and cost for nothing. RAG earns its place only when the answer depends on knowledge outside the model.

## Is RAG always embeddings? No.

> **⭐ Key takeaway —** the single most common misconception: **RAG is not the same as embeddings or vector databases.** RAG means *retrieve, then generate*, and *how* you retrieve is an implementation choice you can change.

- **Embeddings + vector search —** the popular default (notebook 01). Great for matching *meaning* when the user's words differ from the document's.
- **Keyword search (BM25) —** no neural network at all (notebook 02). Often *beats* embeddings for exact identifiers, error codes, names.
- **Hybrid —** both at once, fused (notebook 02). What most production systems actually use.
- **A SQL query, an API call, a plain document lookup —** if it fetches relevant context to put in the prompt, it's RAG.

The demo below proves the point. It retrieves with *nothing but string matching*, no embeddings anywhere, and it's still RAG. We reach for embeddings in notebook 01 because keyword matching misses paraphrases, not because RAG requires them.

## The pipeline, and where this section builds it

Every RAG system is four stages. Data flows in the order below, but this section teaches them in *discovery* order: get retrieval working first, then go back to the chunking decision once you can judge it.

| Stage | What happens | Built in |
|---|---|---|
| **1. Index** | split source docs into chunks, embed, and store them | `01-embeddings-retrieval` (embed + store), `03-chunking` (the splitting decision, revisited last) |
| **2. Retrieve** | given a query, find the most relevant chunks | `01-embeddings-retrieval` (vector), `02-hybrid-and-reranking` (keyword, hybrid, rerank) |
| **3. Augment** | paste the retrieved chunks into the prompt as context | shown in the demo below; used throughout |
| **4. Generate** | the model answers from that context, with citations | the demo below; refined in every notebook |

Why is chunking (`03`) taught *after* retrieval, even though it runs first? Because a chunking choice only makes sense once you've seen retrieval succeed and fail on it. Chunk size is a tradeoff against the embedding model, retrieval precision, and the prompt budget, all of which you meet in `01` and `02`. Then `04-why-rag-fails` is about diagnosing the whole thing when answers are bad, which, spoiler, is almost always a *retrieval* problem rather than a generation one.

First, let's see all four stages at once in the smallest possible form.

## RAG in ~15 lines

A three-document "knowledge base," a toy retriever that just scores by shared words (no embeddings!), and a generation call that answers *only* from what was retrieved. This is a complete RAG system, and everything else in the section makes each stage better.

In [ ]:
# 1. INDEX: a tiny knowledge base. (In a real system these are chunks of your docs.)
DOCS = [
    "The Acme return policy allows refunds within 30 days of purchase with a receipt.",
    "Acme support hours are Monday to Friday, 9am to 6pm Pacific time.",
    "Acme's enterprise plan includes a dedicated account manager and a 99.9% uptime SLA.",
]

# 2. RETRIEVE: score each doc by how many query words it contains. No embeddings --
#    deliberately the dumbest retriever that works, to prove retrieval != embeddings.
def retrieve(query, k=1):
    q_words = set(query.lower().split())
    scored = [(len(q_words & set(d.lower().split())), d) for d in DOCS]
    scored.sort(reverse=True)                       # best overlap first
    return [d for score, d in scored[:k] if score]  # drop zero-overlap docs

# 3. AUGMENT + 4. GENERATE: paste retrieved docs into the prompt, answer only from them.
def rag_answer(question, k=1):
    context = "\n".join(retrieve(question, k)) or "(no relevant documents found)"
    prompt = (
        "Answer the question using ONLY the context below. "
        "If the context does not contain the answer, say you don't know.\n\n"
        f"Context:\n{context}\n\nQuestion: {question}"
    )
    resp = client.chat.completions.create(
        model=MODEL, max_tokens=150,
        messages=[{"role": "user", "content": prompt}],
    )
    return resp.choices[0].message.content

question = "How long do I have to return something to Acme?"
print("RETRIEVED:", retrieve(question))
print("\nANSWER:", rag_answer(question))

Run it. The retriever pulls the return-policy document (it shares the most words with the question), that text goes into the prompt, and the model answers *from it*: "30 days," grounded in a fact it was never trained on.

Now try the failure mode that motivates the entire rest of the section:

In [ ]:
# A question whose answer IS in the knowledge base, but phrased with different words
# than the document uses. Watch the keyword retriever miss it.
q2 = "When can I reach a human at Acme for help?"   # doc says "support hours", not "reach a human"
print("RETRIEVED:", retrieve(q2))
print("\nANSWER:", rag_answer(q2))

Depending on word overlap, the keyword retriever may grab the wrong document or nothing, even though the *support hours* doc answers the question, because the query and the document share almost no literal words ("reach a human" vs "support hours"). **The model can only be as good as what retrieval hands it.** That single failure is why the next notebooks exist:

- **Notebook 01** replaces word-overlap with *embeddings*, so "reach a human" and "support hours" match by meaning.
- **Notebook 02** adds keyword search back as a *complement* (it's still the best tool for exact identifiers) and fuses the two.
- **Notebook 03** goes back to the chunking decision. Now that you've seen retrieval, you can reason about how splitting affects it.
- **Notebook 04** is the discipline of diagnosing exactly this class of bug in a real system.

And you won't tune any of it by vibes: you already installed the eval habit in **section 02** ("measure before you tune"). Every retrieval choice ahead (embedding model, hybrid weights, chunk size) gets judged by a number, not a hunch, and section 04 makes that rigorous for the whole RAG system.

You've now seen the whole loop. The rest of the section is depth on each stage.

## Exercises

1. **Add a document and a query for it.** Append a fourth fact to `DOCS` (say, shipping times), then ask a question it answers. Confirm the retriever surfaces it and the model uses it.
2. **Break it deliberately.** Ask a question whose answer is *not* in `DOCS` at all (e.g. "What is Acme's stock price?"). Confirm the model says it doesn't know rather than inventing an answer; this is the grounding RAG buys you. Then delete the "ONLY the context" instruction from the prompt and watch the behavior change.
3. **Raise `k`.** Call `rag_answer(question, k=3)` so all three docs go into the context. Does answer quality change? This is the precision-vs-recall tension notebooks 01 and 02 formalize: more context means more chances to include the answer, but also more irrelevant text diluting the prompt.
4. **Name the retrieval method.** In one sentence each, describe how you'd swap the toy `retrieve()` for (a) an embedding search, (b) a SQL query against a customer database, (c) a live weather API. All three would still be RAG, so say why.